[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-05-datasets-library.ipynb#scrollTo=b1c2d3e4)

---
# Day 5 · The Datasets Library — Loading, Filtering, Mapping, and Tokenizing
**certified-journeys / huggingface-nlp-certified** · Day 5 · Data Pipelines

> **Goal for today:** By the end of this notebook you can load any HF dataset, inspect its structure, filter and map over it at scale, tokenize efficiently with `batched=True`, and persist processed datasets to disk for reuse.


In [ ]:
%pip install -q datasets transformers torch


## Step 1 · Loading a dataset and inspecting its structure

The `datasets` library provides a unified interface to thousands of datasets hosted on the Hub. `load_dataset` streams or downloads a dataset and returns a `DatasetDict` (or `Dataset` for single splits).

**Reference:** [Datasets quickstart](https://huggingface.co/docs/datasets/quickstart)

Key objects:

| Object | What it is |
|---|---|
| `DatasetDict` | Dict of splits: `{"train": Dataset, "test": Dataset, ...}` |
| `Dataset` | A single split — Arrow-backed columnar table |
| `Features` | Schema: column names, types, class labels |
| `dataset[0]` | First row as a Python dict |
| `dataset[:3]` | First 3 rows as a dict of lists |


In [ ]:
from datasets import load_dataset

# Downloads ~80MB — cached in ~/.cache/huggingface/datasets after first call
dataset = load_dataset("imdb")

print("Type:", type(dataset))
print("Splits:", list(dataset.keys()))
print()
print("Train split:")
print("  Num rows:", len(dataset["train"]))
print("  Features:", dataset["train"].features)
print()
print("Test split:")
print("  Num rows:", len(dataset["test"]))


In [ ]:
# Inspect a single example
example = dataset["train"][0]
print("Keys:", list(example.keys()))
print("Label:", example["label"])  # 0 = negative, 1 = positive
print("Text preview:", example["text"][:200], "...")

# Slice — returns dict of lists
first_three = dataset["train"][:3]
print("\nLabels for first 3:", first_three["label"])


### What just happened?
- `load_dataset('imdb')` returned a `DatasetDict` with `train` (25 000 rows) and `test` (25 000 rows) splits.
- **Arrow-backed storage** means datasets live in a memory-mapped file on disk — you can iterate over datasets larger than RAM without loading everything into memory.
- `features` describes the schema: `text` is a string column, `label` is a `ClassLabel` with names `["neg", "pos"]`.
- Single-row access (`dataset[0]`) returns a dict; slicing (`dataset[:3]`) returns a dict of lists — consistent with how `map` operates.


## Step 2 · Filtering with `dataset.filter()`

`filter` applies a boolean function row-by-row (or in batch) and keeps only rows where it returns `True`.

```
new_dataset = dataset.filter(fn)  # fn(example) → bool
new_dataset = dataset.filter(fn, batched=True)  # fn(batch_dict) → list[bool]
```

Under the hood, `filter` streams through the Arrow table and writes a new cached Arrow file — it does **not** modify the original dataset in place.


In [ ]:
train_data = dataset["train"]

# Keep only positive reviews (label == 1)
positive_train = train_data.filter(lambda example: example["label"] == 1)

print("Original train size :", len(train_data))
print("Positive-only size  :", len(positive_train))
print("Labels in filtered  :", set(positive_train["label"]))  # should be {1}


In [ ]:
# More complex filter: keep only long reviews (> 200 words)
# Use batched=True for a significant speed-up on large datasets
def is_long_review(batch):
    # batch is a dict of lists — process the whole batch at once
    return [len(text.split()) > 200 for text in batch["text"]]

long_reviews = train_data.filter(is_long_review, batched=True)
print("Long review count (>200 words):", len(long_reviews))

# Verify
sample_length = len(long_reviews[0]["text"].split())
print("First example word count:", sample_length)


### What just happened?
- `filter(lambda ...)` is the simplest form — the lambda receives one row (a dict) and returns `True/False`.
- `filter(..., batched=True)` receives a **dict of lists** (a batch of rows) and must return a list of booleans — this is much faster for complex operations because it avoids Python function-call overhead per row.
- **Filtering is lazy-cached**: the result is written to a new Arrow file under `~/.cache/huggingface/datasets`; calling the same filter again uses the cache.
- The original `train_data` is unchanged — `filter` always returns a new `Dataset`.


## Step 3 · Mapping and tokenizing with `dataset.map()`

`map` transforms every example — you can add columns, remove columns, or convert text to token IDs. The `batched=True` flag is the key performance lever.

| Mode | Function signature | Speed |
|---|---|---|
| `batched=False` (default) | `fn(example: dict) → dict` | Slow — one Python call per row |
| `batched=True` | `fn(batch: dict of lists) → dict of lists` | Fast — 10x+ speedup |

When tokenizing, always use `batched=True` — HF tokenizers are parallelised internally and can process thousands of strings per call.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(batch):
    # batch["text"] is a list of strings when batched=True
    return tokenizer(
        batch["text"],
        truncation=True,   # clip sequences longer than max_length
        max_length=128,    # keep short for speed in this demo
        padding=False,     # don't pad here — DataCollator handles it during training
    )

# Apply across the full train split
# remove_columns drops the original 'text' column we no longer need
tokenized_train = train_data.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

print("Original columns:", train_data.column_names)
print("Tokenized columns:", tokenized_train.column_names)
print("First row input_ids (first 10):", tokenized_train[0]["input_ids"][:10])
print("First row length:", len(tokenized_train[0]["input_ids"]))


In [ ]:
# You can also add computed columns with map — example: add a word-count column
def add_word_count(batch):
    return {"word_count": [len(text.split()) for text in batch["text"]]}

# Apply to original dataset (not tokenized — we need 'text' column)
with_wc = train_data.map(add_word_count, batched=True)

print("Columns now:", with_wc.column_names)
print("Word counts for first 3:", with_wc[:3]["word_count"])

# Compute average word count
import statistics
avg_wc = statistics.mean(with_wc["word_count"])
print(f"Average review length: {avg_wc:.0f} words")


### What just happened?
- `map(tokenize_fn, batched=True)` passes a **chunk** of rows at a time to your function — the HF tokenizer can process a whole list of strings in one call, leveraging Rust-based parallelism internally.
- **`remove_columns`** drops the raw text column after tokenization — keeps the dataset lean since we no longer need it for training.
- `map` can **add new columns** by returning a dict with new keys — existing columns are preserved unless explicitly removed.
- The output is cached; if you re-run the same `map` call, it returns the cached result instantly.


## Step 4 · Shuffle and select a subset

During rapid prototyping you rarely need all 25 000 training examples. Two dataset operations help:

- **`shuffle(seed=42)`** — randomly permutes row order; `seed` ensures reproducibility
- **`select(range(N))`** — takes the first N rows from the current order

Always shuffle **before** select so your subset is representative, not just the first N rows in the original ordering.


In [ ]:
# Shuffle with a fixed seed for reproducibility, then take 1000 examples
small_train = train_data.shuffle(seed=42).select(range(1000))

print("Full train size :", len(train_data))
print("Subset size     :", len(small_train))

# Verify the subset is balanced (roughly 50/50 pos/neg after shuffle)
pos_count = sum(1 for label in small_train["label"] if label == 1)
neg_count = len(small_train) - pos_count
print(f"Positive: {pos_count}  Negative: {neg_count}")


In [ ]:
# Now tokenize the small subset
tokenized_small = small_train.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
)

# Set PyTorch format so the dataset returns tensors instead of lists
tokenized_small.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Verify format
row = tokenized_small[0]
print("input_ids type:", type(row["input_ids"]))
print("input_ids shape:", row["input_ids"].shape)
print("label          :", row["label"])


### What just happened?
- `shuffle(seed=42)` permutes the row order deterministically — the same seed always produces the same permutation, so experiments are reproducible.
- `select(range(1000))` is O(1) — it just records which row indices to expose without copying data.
- **`set_format(type='torch')`** makes the dataset behave like a PyTorch `Dataset` — indexing returns tensors, which is what `DataLoader` expects during training.
- The combination `shuffle().select()` is the standard pattern for creating dev subsets during rapid iteration.


## Step 5 · Saving to disk and reloading

Tokenization on large datasets can take minutes. `save_to_disk` writes the processed Arrow dataset to a directory; `load_from_disk` reloads it instantly — no recomputation needed.

| Method | Use case |
|---|---|
| `save_to_disk(path)` | Persist processed dataset for reuse |
| `load_from_disk(path)` | Reload without re-running map/filter |
| `push_to_hub(repo_id)` | Share on the HF Hub |
| `dataset.to_parquet(path)` | Export as Parquet for non-HF pipelines |


In [ ]:
import os
from datasets import load_from_disk

# Reset format before saving — saved datasets should be format-agnostic
tokenized_small.reset_format()

save_path = "/tmp/imdb_tokenized_1k"
tokenized_small.save_to_disk(save_path)

# Check what was saved
saved_files = os.listdir(save_path)
print("Files in saved dataset directory:", saved_files)

# Reload from disk
reloaded = load_from_disk(save_path)
print("Reloaded type     :", type(reloaded).__name__)
print("Reloaded num rows :", len(reloaded))
print("Reloaded columns  :", reloaded.column_names)
print("First row label   :", reloaded[0]["label"])


In [ ]:
# Bonus: save a full DatasetDict (multiple splits)
from datasets import DatasetDict

# Build a small train/test DatasetDict
small_test = dataset["test"].shuffle(seed=42).select(range(200))
tokenized_test = small_test.map(tokenize_fn, batched=True, remove_columns=["text"])

full_tokenized = DatasetDict({
    "train": tokenized_small,
    "test" : tokenized_test,
})

dict_save_path = "/tmp/imdb_tokenized_dict"
full_tokenized.save_to_disk(dict_save_path)

reloaded_dict = load_from_disk(dict_save_path)
print("Reloaded DatasetDict splits:", list(reloaded_dict.keys()))
print("  Train rows:", len(reloaded_dict["train"]))
print("  Test rows :", len(reloaded_dict["test"]))


### What just happened?
- `save_to_disk` writes Arrow IPC files (`data-00000-of-00001.arrow`), a `dataset_info.json` with schema metadata, and a `state.json` for format tracking — everything needed to reconstruct the dataset.
- `load_from_disk` reads the Arrow files directly — it's near-instantaneous and returns the same `Dataset`/`DatasetDict` type you saved.
- **Always reset format before saving** — the PyTorch tensor format is a runtime view, not stored on disk; re-apply `set_format` after reloading.
- Saving a `DatasetDict` creates a subdirectory per split inside the target path.


In [ ]:
# Challenge: Load the 'ag_news' dataset, keep only category 1 (Sports) from
# the training split, tokenize with distilbert-base-uncased (max_length=64,
# batched=True), then save to /tmp/ag_news_sports and reload it.
# Print: number of rows saved, column names, and the text of the first example.
#
# Hint: ag_news has columns 'text' and 'label'. Category 1 = Sports.
# Don't forget to remove_columns=["text"] after tokenization.
#
# Scaffold:
# from datasets import load_dataset, load_from_disk
# from transformers import AutoTokenizer
#
# dataset = load_dataset("ag_news")
# sports = dataset["train"].filter(...)  # TODO: keep label == 1
#
# tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# def tokenize_fn(batch):
#     return tokenizer(...)  # TODO: fill in
#
# tokenized = sports.map(...)  # TODO: fill in
# tokenized.save_to_disk("/tmp/ag_news_sports")
#
# reloaded = load_from_disk("/tmp/ag_news_sports")
# print(...)

# Your solution here


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `load_dataset` | Returns a `DatasetDict`; Arrow-backed, memory-mapped |
| `dataset.features` | Schema: column names, types, ClassLabel definitions |
| `dataset.filter` | Returns new Dataset with only rows where fn returns True |
| `batched=True` in map/filter | Passes lists not scalars — 10x+ faster for tokenization |
| `remove_columns` in map | Drop raw text columns after tokenization to keep dataset lean |
| `shuffle(seed=42).select(N)` | Reproducible subset — always shuffle before select |
| `set_format(type='torch')` | Runtime view — datasets return tensors; reset before saving |
| `save_to_disk` / `load_from_disk` | Persist processed datasets to avoid re-running map each session |

> **Tip:** Use `batched=True` in `dataset.map()` — it passes lists of examples to your function instead of one at a time, making tokenization 10x faster on large datasets.

---
## What's next
**Day 6** → The Evaluate Library — Metrics After Fine-Tuning: computing F1, BLEU, and ROUGE scores programmatically to measure model quality.

Mark Day 5 complete in your [tracker](../index.html).
